# Phase 1 + 2: FP32 baselines and compression sweep on Kaggle / Colab

Trains `tiny_cnn`, `m5`, `logmel_cnn` and `effnet_b0` on CirCor with patient-level 5-fold CV and 3 seeds (60 runs) using the official CinC 2022 scorer, then applies every compression condition (INT8 PTQ per-channel / per-tensor, INT8 QAT, INT4 weight-only simulation, dynamic INT8, ONNX Runtime INT8) to each checkpoint.

**Kaggle setup:** Settings -> Accelerator: GPU (T4 x2 or P100); Internet: ON. Upload `dist/pcg_code.zip` as a Kaggle Dataset named `pcg-code` and attach it under *Input*.

**Colab setup:** Runtime -> Change runtime type -> GPU. Upload `pcg_code.zip` via the file panel (or mount Drive and set `CODE_ZIP`).

Runs are resumable: re-running the training cell skips finished (model, fold, seed) combinations, so a 12 h Kaggle limit is not a problem. Download `runs.zip` at the end and unzip it into the project's `runs/` folder on your laptop.

In [ ]:
import os, glob, subprocess, sys, zipfile
from pathlib import Path

ON_KAGGLE = Path('/kaggle/working').exists()
WORK = Path('/kaggle/working') if ON_KAGGLE else Path('/content')
PROJ = WORK / 'pcg_project'

# Where the code zip is. Kaggle: attached dataset. Colab: uploaded file or Drive path.
candidates = glob.glob('/kaggle/input/**/pcg_code.zip', recursive=True) + glob.glob(str(WORK / 'pcg_code.zip')) + glob.glob('/content/drive/MyDrive/**/pcg_code.zip', recursive=True)
CODE_ZIP = os.environ.get('CODE_ZIP') or (candidates[0] if candidates else None)
assert CODE_ZIP, 'pcg_code.zip not found: upload it (see the markdown cell above)'
PROJ.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP) as z:
    z.extractall(PROJ)
os.chdir(PROJ)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'soundfile', 'captum', 'onnx', 'onnxruntime', 'torchao'], check=True)
import torch, torchvision
print('project:', PROJ, '| torch', torch.__version__, '| torchvision', torchvision.__version__, '| cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
# CirCor v1.0.3 from the AWS Open Data mirror (~586 MB, usually 1-3 min on Kaggle).
# If you attached a Kaggle dataset that already contains `training_data/`, it is used instead.
raw = PROJ / 'data' / 'raw'
raw.mkdir(parents=True, exist_ok=True)
pre = [p for p in glob.glob('/kaggle/input/**/training_data', recursive=True) if Path(p).is_dir()]
if pre and not (raw / 'circor-heart-sound-1.0.3').exists():
    os.symlink(Path(pre[0]).parent, raw / 'circor-heart-sound-1.0.3', target_is_directory=True)
    print('using attached dataset:', pre[0])
else:
    subprocess.run([sys.executable, 'scripts/download_circor_s3.py'], check=True)
subprocess.run([sys.executable, 'scripts/p0_verify_data.py'], check=True)

In [ ]:
# Phase 1: the full FP32 grid. Edit MODELS/FOLDS/SEEDS to split across sessions if needed.
MODELS = ['tiny_cnn', 'm5', 'logmel_cnn', 'effnet_b0']
FOLDS = ['0', '1', '2', '3', '4']
SEEDS = ['0', '1', '2']
EPOCHS = '30'
RUNS = str(WORK / 'runs')
cmd = [sys.executable, 'scripts/p1_train.py', '--model', *MODELS, '--folds', *FOLDS, '--seeds', *SEEDS,
       '--epochs', EPOCHS, '--runs-dir', RUNS, '--workers', '2']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Phase 2: compression sweep on every finished checkpoint (CPU-bound; INT8 kernels are CPU kernels).
# QAT fine-tuning runs on CPU here too; lower --qat-epochs for the 4M-param effnet_b0 if the session is short.
cmd = [sys.executable, 'scripts/p2_quantize.py', '--model', *MODELS, '--folds', *FOLDS, '--seeds', *SEEDS,
       '--runs-dir', RUNS, '--qat-epochs', '5', '--threads', '4']
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
# Summarise and package results for download (runs.zip: checkpoints, ONNX files, per-recording probabilities, metrics).
import json, pandas as pd
rows = []
for f in glob.glob(f'{RUNS}/*/fold*/seed*/patient_test.json'):
    p = Path(f)
    rows.append({'model': p.parts[-4], 'fold': p.parts[-3], 'seed': p.parts[-2]} | json.loads(p.read_text()))
df = pd.DataFrame(rows)
display(df.groupby('model')[['weighted_accuracy', 'auroc', 'present_recall', 'cost']].agg(['mean', 'std']).round(3))
if (Path(RUNS) / 'p2_summary.csv').exists():
    q = pd.read_csv(Path(RUNS) / 'p2_summary.csv')
    display(q.groupby(['model', 'cond'])[['weighted_accuracy', 'auroc', 'present_recall', 'cost']].mean().round(3))
out = WORK / 'runs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in Path(RUNS).rglob('*'):
        if f.is_file():
            z.write(f, f.relative_to(WORK).as_posix())
print('download:', out, f'{out.stat().st_size/1e6:.1f} MB')